# Cvičení 3

In [1]:
:opt no-lint

Řekněme, že chceme realizovat kartézský součin. Mohli bychom využít list comprehension:

In [2]:
allPairs :: [a] -> [b] -> [(a, b)]
allPairs xs ys = [ (x, y) | x <- xs, y <- ys ]

In [3]:
allPairs [1,2] [3,4] == [(1,3),(1,4),(2,3),(2,4)]

True

In [4]:
allPairs [] [3, 4]

[]

In [5]:
allPairs [1, 2] []

[]

Jak bychom takovou věc realizovali bez list comprehension?

**Otázka:** Doplňte rekurzivní funkci níže.

In [6]:
allPairs :: [a] -> [b] -> [(a, b)]

allPairs [] _ = []
allPairs (x:xs) ys = pairWith x ys ++ allPairs xs ys
    where
        pairWith :: a -> [b] -> [(a, b)]
        pairWith _ [] = []
        pairWith x (y:ys) = (x, y) : pairWith x ys

In [7]:
allPairs [1,2] [3,4] == [(1,3),(1,4),(2,3),(2,4)]

True

In [8]:
cardRanks :: [Int]
cardRanks = [2, 3, 4, 5]

cardSuits :: [Char]
cardSuits = ['H', 'D', 'C', 'S']  -- hearts ♥, diamonds ♦, clubs ♣, spades ♠

allPairs cardRanks cardSuits

[(2,'H'),(2,'D'),(2,'C'),(2,'S'),(3,'H'),(3,'D'),(3,'C'),(3,'S'),(4,'H'),(4,'D'),(4,'C'),(4,'S'),(5,'H'),(5,'D'),(5,'C'),(5,'S')]

**Otázka:** Zadefinujte datový typ vhodný pro reprezentaci karty a vytvořte pro něj vhodnou implementaci typové třídy `Show`, aby se karty vypisovaly jako `2H`, `3D` atp.

In [9]:
data Card = Card Int Char

instance Show Card where
    show (Card i c) = show i ++ [c]

In [10]:
Card 2 'H'

2H

Varianta funkce `allPairs`, která vytváří karty:

In [11]:
allCards :: [Int] -> [Char] -> [Card]

allCards [] _ = []
allCards (x:xs) ys = pairWith x ys ++ allCards xs ys
    where
        pairWith :: Int -> [Char] -> [Card]
        pairWith _ [] = []
        pairWith a (b:bs) = Card a b : pairWith a bs

In [12]:
allCards cardRanks cardSuits

[2H,2D,2C,2S,3H,3D,3C,3S,4H,4D,4C,4S,5H,5D,5C,5S]

**Otázka:** Zabstrahujte funkce `allCards` a `allPairs` do obecné HOF `allCombs`, která s pomocí dodané funkce vytváří kartézský součin dvou seznamů libovolného typu. Pak pomocí této funkce vyjádřete `allPairs` i `allCards`.

In [13]:
allCombs :: (a -> b -> c) -> [a] -> [b] -> [c]

allCombs _ [] _ = []
allCombs f (x:xs) ys = combine x ys ++ allCombs f xs ys
    where
        combine _ [] = []
        combine x (y:ys) = f x y : combine x ys

In [14]:
-- Doplňte i tyto funkce
allPairs :: [a] -> [b] -> [(a, b)]
allPairs = allCombs (,)

allCards :: [Int] -> [Char] -> [Card]
allCards = allCombs Card

**Otázka:** Tato funkce umí generovat kombinace ze dvou seznamů.  Nyní napište funkci `allCombs3`, která bude dělat totéž pro tři vstupní seznamy.

In [15]:
allCombs3 :: (a -> b -> c -> d) -> [a] -> [b] -> [c] -> [d]

allCombs3 f xs ys zs =
  let combs2 = allCombs f xs ys
   in allCombs (\g z -> g z) combs2 zs

In [16]:
:t allCombs (\a b c -> (a, b, c)) ['x'] [True]

allCombs (\a b c -> (a, b, c)) ['x'] [True] :: forall {c}. [c -> (Char, Bool, c)]

In [17]:
allCombs3 (,,) [1,2] [3,4] [5,6] == [ (x, y, z) | x <- [1,2], y <- [3,4], z <- [5,6] ]

True

Existuje způsob, jak tohle pěkně zobecnit.

In [18]:
combStep :: [a -> b] -> [a] -> [b]  -- m (a -> b) -> m a -> m b

combStep [] _ = []
combStep (f:fs) ys = applyAll f ys ++ combStep fs ys
  where
    applyAll :: (a -> b) -> [a] -> [b]
    applyAll _ [] = []
    applyAll f' (v:vs) = (f' v) : applyAll f' vs

Srovnejme s `allPairs`! V podstatě to dělá totéž – kartézský součin funkcí s hodnotami, což vede na aplikaci těch funkcí.

In [19]:
combStep [(+1)] [100, 200]
combStep [(+1), (*2)] [100, 200]
combStep [(+1), (*2), (/5)] [100, 200]

[101,201]

[101,201,200,400]

[101.0,201.0,200.0,400.0,20.0,40.0]

In [20]:
cardRanks     -- Pro připomenutí
mkPair = (,)  -- Datový konstruktor je funkce!
:t mkPair 2   -- Vytvoří funkci, která po aplikaci na y vytvoří dvojici (2, y).
:t map mkPair cardRanks  -- Vytvoří seznam takových funkcí pro každý prvek z cardRanks:
-- [ (\y -> (2, y)), (\y -> (3, y)), (\y -> (4, y)), (\y -> (5, y)) ]

[2,3,4,5]

mkPair 2 :: forall {a} {b}. Num a => b -> (a, b)

map mkPair cardRanks :: forall {b}. [b -> (Int, b)]

In [21]:
combStep (map mkPair cardRanks) [100, 200]

[(2,100),(2,200),(3,100),(3,200),(4,100),(4,200),(5,100),(5,200)]

In [22]:
allCombs :: (a -> b -> c) -> [a] -> [b] -> [c]

allCombs f xs ys = combStep (map f xs) ys

In [23]:
allCombs Card cardRanks cardSuits

[2H,2D,2C,2S,3H,3D,3C,3S,4H,4D,4C,4S,5H,5D,5C,5S]

**Otázka:** Realizujte s využitím `combStep` funkci `allCombs3`.

In [24]:
allCombs3 :: (a -> b -> c -> d) -> [a] -> [b] -> [c] -> [d]

allCombs3 f xs ys zs = combStep second zs
    where
        second = combStep first ys
        first = map f xs

In [25]:
allCombs3 (,,) [1,2] [3,4] [5,6] == [ (x, y, z) | x <- [1,2], y <- [3,4], z <- [5,6] ]

True

#### Kombinace jako *efekt* seznamu

Doteď jsme seznam chápali jako obyčejnou datovou strukturu, ve které jen bydlí nějaké hodnoty. \
Zkusme jiný pohled: hodnota typu `[a]` může znamenat „**výpočet**, který může vrátit **více možných výsledků** typu a“ – seznamem můžeme modelovat **nedeterministický výpočet**.

Pak:
- `map f xs` znamená: transformuj všechny možné výsledky,
- `allCombs f xs ys` znamená: vezmi všechny možnosti z `xs`, všechny možnosti z `ys`, a zkombinuj je _(co by, kdyby)_,
- `combStep` znamená: máme seznam funkcí a seznam možných vstupů, aplikuj všechny funkce na všechny vstupy.

`combStep` tedy vyjadřuje **aplikaci** seznamu.

**Otázka:** Implementujte funkci `applyNondet`, která vezme „nedeterministickou funkci“ tvaru `(a -> [b])` a umožní ji použít na každý předchozí výsledek předchozího „nedeterministického výpočtu“ `[a]`, čímž vznikne série nových nedeterministických výsledků `[b]`. Hint: využijte `concat` a `map`.

In [26]:
applyNondet :: (a -> [b]) -> [a] -> [b]

applyNondet _ [] = []
applyNondet f xs = concat (map f xs)

In [27]:
generation x = [x ++ "_1", x ++ "_2", x ++ "_3"]

applyNondet generation ["bunny"]
applyNondet generation (applyNondet generation ["bunny"])
applyNondet generation (applyNondet generation (applyNondet generation ["bunny"]))

["bunny_1","bunny_2","bunny_3"]

["bunny_1_1","bunny_1_2","bunny_1_3","bunny_2_1","bunny_2_2","bunny_2_3","bunny_3_1","bunny_3_2","bunny_3_3"]

["bunny_1_1_1","bunny_1_1_2","bunny_1_1_3","bunny_1_2_1","bunny_1_2_2","bunny_1_2_3","bunny_1_3_1","bunny_1_3_2","bunny_1_3_3","bunny_2_1_1","bunny_2_1_2","bunny_2_1_3","bunny_2_2_1","bunny_2_2_2","bunny_2_2_3","bunny_2_3_1","bunny_2_3_2","bunny_2_3_3","bunny_3_1_1","bunny_3_1_2","bunny_3_1_3","bunny_3_2_1","bunny_3_2_2","bunny_3_2_3","bunny_3_3_1","bunny_3_3_2","bunny_3_3_3"]

---

In [28]:
newtype Seed = Seed Integer
  deriving (Eq, Show)

mkSeed :: Integer -> Seed
mkSeed = Seed
unSeed :: Seed -> Integer
unSeed (Seed s) = s

type Gen a = Seed -> (a, Seed)

rand :: Gen Integer
rand (Seed s) = (s', Seed s')
  where
    s' = (s * 16807) `mod` 0x7FFFFFFF

-- vezme mapovací funkci a -> b a libovolný generátor Gen a a vytvoří nový generátor Gen b,
-- uvnitř provede to, co dělá Gen a, ale namapuje na výslednou hodnotu tu funkci.
generalA :: (a -> b) -> Gen a -> Gen b
generalA f gen = \s -> 
  let (rnd, s') = gen s 
  in  (f rnd, s')

-- vezme dva generátory Gen a, Gen b a konstrukční funkci a -> b -> c a vytvoří generátor,
-- ten spustí generátory postupně za sebou (přičemž zase váže vnitřní stav) a z obou postupně 
-- získaných hodnot pomocí funkce složí libovolný výsledek.
generalB :: Gen a -> Gen b -> (a -> b -> c) -> Gen c
generalB genA genB f = \s ->
  let (r1, s2) = genA s
      (r2, s3) = genB s2
   in (f r1 r2, s3)

-- vyrobí konstantní generátor – v podstatě materializuje zadanou konstantu do podoby generátoru.
mkGen :: a -> Gen a
mkGen x s = (x, s)

-- vezme seznam generátorů a vytvoří z něj jeden generátor seznamu,
-- který postupně vyhodnotí všechny generátory, přičemž provázává vnitřní stav 
-- a nakonec vrací spolu s vzniklým seznamem i ten výsledný stav.
repRandom :: [Gen a] -> Gen [a]
repRandom [] = mkGen []
repRandom (g:gs) = \s -> 
  let (rnd, s')   = g s
      (next, s'') = repRandom gs s'
  in  (rnd : next, s'')

-- „pokračuj na základě funkce“ – vezme generátor a pak nějakou funkci, která řekne, jak má vzniknout 
-- nový generátor podle výsledku předchozího. Operace pak zase zajistí provázání stavu: vezme výsledný stav 
-- prvního generování a strčí ho do toho nového generátoru.
continueWith :: Gen a -> (a -> Gen b) -> Gen b
continueWith g f = \s ->
  let (rnd, s') = g s
      newGen    = f rnd
   in newGen s'

---
```haskell
data Maybe a = Nothing | Just a
```

In [29]:
link :: Maybe a -> (a -> Maybe b) -> Maybe b  -- ~ continueWith
link Nothing _ = Nothing
link (Just x) f = f x

mkMaybe :: a -> Maybe a  -- ~ mkGen
mkMaybe = Just

transMaybe :: (a -> b) -> Maybe a -> Maybe b  -- ~ generalA
transMaybe f mx =
    mx `link` \x -> mkMaybe (f x)

yLink :: (a -> b -> c) -> Maybe a -> Maybe b -> Maybe c  -- ~ generalB
yLink f ma mb =
    ma `link` \a ->
    mb `link` \b ->
    mkMaybe (f a b)

combine :: Maybe (Maybe a) -> Maybe a
combine mmx = mmx `link` id

Povšimněte si, že u `Maybe` je `transMaybe` i `yLink` operace vyjádřená pomocí `link`, zatímco u generátorů mám ručně provázané stavy (seeds). Toho bychom se mohli zbavit:

In [30]:
generalA :: (a -> b) -> Gen a -> Gen b
generalA f gx = 
    gx `continueWith` \x -> 
    mkGen (f x)

**Otázka:** Implementujte `generalB` pomocí `continueWith`:

In [31]:
generalB :: Gen a -> Gen b -> (a -> b -> c) -> Gen c

generalB ga gb f =
    ga `continueWith` \a ->
    gb `continueWith` \b ->
    mkGen (f a b)

**Otázka:** Implementujte `repRandom` pomocí `continueWith` (nebo případně `repMaybe` pomocí `link`):

In [32]:
repRandom :: [Gen a] -> Gen [a]
repMaybe :: [Maybe a] -> Maybe [a]

repRandom [] = mkGen []
repRandom (g:gs) =
    g `continueWith` \gv ->
    repRandom gs `continueWith` \iv ->
    mkGen $ gv : iv

repMaybe [] = mkMaybe []
repMaybe (m:ms) =
    m `link` \mv ->
    repMaybe ms `link` \iv ->
    mkMaybe $ mv : iv

In [33]:
repRandom (replicate 4 rand) (mkSeed 1)

([16807,282475249,1622650073,984943658],Seed 984943658)

In [34]:
repMaybe [Just 1, Just 2, Just 3]
repMaybe [Just 1, Just 2, Nothing, Just 8]

Just [1,2,3]

Nothing

Čas definovat minimální množinu obdobně fungujících operací.
> Let’s call it a monad! You know how companies these days have been giving themselves nonsensical names that allow them to completely define their brand without competing with their customers’ preconceived notions of what common words mean? We’re doing the same thing here.

```haskell
class Monad m where
    -- link         :: Maybe a  -> (a -> Maybe b ) -> Maybe b
    -- continueWith :: Gen   a  -> (a -> Gen   b ) -> Gen   b
    -- flip applyNondet ::  [a] -> (a ->      [b]) ->      [b]
    (>>=)    :: Monad m => m a -> (a -> m b)     -> m b  -- čteme "bind"

    -- mkMaybe ::        a -> Maybe a
    -- mkGen   ::        a -> Gen   a
    return :: Monad m => a -> m a
```

Ostatní operace, které jdou ale odvodit:
```haskell
-- transMaybe, generalA, –
fmap   :: (Functor m)     => (a -> b) -> m a -> m b
-- yLink, generalB, allCombs
liftA2 :: (Applicative m) => (a -> b -> c) -> m a -> m b -> m c  -- taky liftM2
-- —, –, allCombs3
liftA3 :: (Applicative m) => (a -> b -> c -> d) -> m a -> m b -> m c -> m d  -- taky liftM3

-- –, –, combStep
<*>    :: (Applicative m) => m (a -> b) -> m a -> m b  -- čteme "ap" nebo "apply"
-- –, combine, –
join   :: (Monad m)       => m (m a) -> m a
-- repRandom, –, –
sequence :: (Monad m)     => [m a] -> m [a]
```

In [35]:
class MyMonad m where
    bind    :: m a -> (a -> m b) -> m b
    mReturn :: a -> m a

In [36]:
apply :: MyMonad m => m (a -> b) -> m a -> m b
apply mf ma =
    mf `bind` \f ->
    ma `bind` \a ->
    mReturn (f a)

In [37]:
fmap' :: MyMonad m => (a -> b) -> m a -> m b
fmap' f ma = 
    ma `bind` \a ->
    mReturn $ f a

fmap' f ma =
    (mReturn f) `apply` ma

In [38]:
liftA2' :: MyMonad m => (a -> b -> c) -> m a -> m b -> m c

liftA2' f ma mb =
    ma `bind` \a ->
    mb `bind` \b ->
    mReturn (f a b)

liftA2' f ma mb = ((mReturn f) `apply` ma) `apply` mb

In [39]:
liftA3' :: MyMonad m => (a -> b -> c -> d) -> m a -> m b -> m c -> m d
liftA3' f ma mb mc =
    ma `bind` \a ->
    mb `bind` \b ->
    mc `bind` \c ->
    mReturn (f a b c)

liftA3' f ma mb mc = (((mReturn f) `apply` ma) `apply` mb) `apply` mc

In [40]:
instance MyMonad Maybe where
    bind :: Maybe a -> (a -> Maybe b) -> Maybe b
    bind Nothing _ = Nothing
    bind (Just v) f = f v

    mReturn v = Just v

In [41]:
instance MyMonad [] where
    bind :: [a] -> (a -> [b]) -> [b]
    bind xs f = concat $ map f xs

    mReturn :: a -> [a]
    mReturn x = [x]

In [42]:
generation x = [x ++ "_1", x ++ "_2", x ++ "_3"]

["bunny"] `bind` generation
["bunny"] `bind` generation `bind` generation
["bunny"] `bind` generation `bind` generation `bind` generation

["bunny_1","bunny_2","bunny_3"]

["bunny_1_1","bunny_1_2","bunny_1_3","bunny_2_1","bunny_2_2","bunny_2_3","bunny_3_1","bunny_3_2","bunny_3_3"]

["bunny_1_1_1","bunny_1_1_2","bunny_1_1_3","bunny_1_2_1","bunny_1_2_2","bunny_1_2_3","bunny_1_3_1","bunny_1_3_2","bunny_1_3_3","bunny_2_1_1","bunny_2_1_2","bunny_2_1_3","bunny_2_2_1","bunny_2_2_2","bunny_2_2_3","bunny_2_3_1","bunny_2_3_2","bunny_2_3_3","bunny_3_1_1","bunny_3_1_2","bunny_3_1_3","bunny_3_2_1","bunny_3_2_2","bunny_3_2_3","bunny_3_3_1","bunny_3_3_2","bunny_3_3_3"]

---

In [43]:
newtype Gen a = Gen { runGen :: Seed -> (a, Seed) }
evalGen :: Gen a -> Seed -> a
evalGen (Gen gen) s = fst $ gen s

rand :: Gen Integer
rand = Gen rand'
    where rand' :: Seed -> (Integer, Seed)
          rand' (Seed s) = (s', Seed s')
              where s' = (s * 16807) `mod` 0x7FFFFFFF

In [44]:
instance Monad Gen where
    (>>=) :: Gen a -> (a -> Gen b) -> Gen b
    (>>=) (Gen gen) f = Gen $ \s ->
        let (rnd, s') = gen s
            (Gen g2) = f rnd
         in g2 s'

    return r = Gen $ \s -> (r, s)

instance Applicative Gen where
    pure = return
    (<*>) mf ma =
        mf >>= \f ->
        ma >>= \a ->
        return (f a)

instance Functor Gen where
    fmap f ma = return f <*> ma

In [45]:
fiveRands = do
    r1 <- rand
    r2 <- rand
    r3 <- rand
    r4 <- rand
    r5 <- rand
    return [r1, r2, r3, r4, r5]

In [46]:
evalGen fiveRands (Seed 1)

[16807,282475249,1622650073,984943658,1144108930]

In [47]:
randLetter = do
    r <- rand
    r2 <- rand
    let toLetter n = (toEnum $ fromIntegral $ n `mod` 26 + 97) :: Char
    let ltr = toLetter r
    let ltr2 = toLetter r2
    return (r, r2)

In [48]:
:t randLetter

randLetter :: Gen (Integer, Integer)

In [49]:
evalGen randLetter (Seed 598)

(10050586,1416474436)

In [50]:
import Data.Time.Clock.POSIX (getPOSIXTime)
import Control.Monad (replicateM)

printRndNums n = do
    seed <- Seed . round . (* 1000) <$> getPOSIXTime
    let gens = sequence $ replicate n rand
    let val = evalGen gens seed
    mapM_ print val

In [51]:
printRndNums 3

2084084865
1756043485
955091674

## IO

In [52]:
-- jupyter boilerplate
import System.IO
import GHC.IO.Handle
import Control.Exception
import System.Directory

withStdin :: String -> IO a -> IO a
withStdin s action = do
    writeFile "/tmp/stdin.txt" s
    finally
        (withFile "/tmp/stdin.txt" ReadWriteMode
            (\h -> do
                  stdin' <- hDuplicate stdin
                  hDuplicateTo h stdin
                  finally action (hDuplicateTo stdin' stdin)            
            )
        )
        (removeFile "/tmp/stdin.txt")

V předchozí části jsme si řekli, že monády jsou způsob, jak **bezpečně skládat výpočty nesoucí nějaký kontext**. U `Maybe` tím kontextem bylo možné selhání, u `Gen` vnitřní stav generátoru. Nyní se podíváme na monádu, která reprezentuje vstupně/výstupní akce: `IO`.

Haskell je *pure* jazyk, takže nemůžeme jen tak „někde bokem“ měnit svět. Operace jako čtení ze vstupu, výpis na obrazovku nebo práce se soubory proto nejsou obyčejné hodnoty, ale **akce**. Hodnota typu `IO a` je v podstatě popis akce, která po spuštění může komunikovat se světem a *nakonec může vyrobit hodnotu* typu `a`.

Například:
- `putStrLn "Ahoj"` má typ `IO ()` – je to akce, která něco vypíše, ale nezajímá nás žádná zajímavá výsledná hodnota,
- `getLine` má typ `IO String` – je to akce, která přečte řádek a vrátí `String`,
- `print 123` má opět typ `IO ()`.

V běžném Haskell programu bývá „vstupním bodem“ akce `main :: IO ()`. V notebooku nebo v GHCi ale můžete spouštět libovolné `IO` akce „přímo“.

In [53]:
:t putStrLn
:t getLine
:t print
:t return @IO

putStrLn :: String -> IO ()

getLine :: IO String

print :: forall a. Show a => a -> IO ()

return @IO :: forall a. a -> IO a

Povšimněte si typu `return`. Stejně jako u předchozích monád je to operace, která vezme obyčejnou hodnotu a **vloží ji do kontextu**. Připomínám tedy, že `return` **není** nějaké kouzelné klíčové slovo, které by něco ukončovalo! JE TO STUPIDNÍ NÁZEV. You know what? Radši používejte `pure`.

In [54]:
return "ahoj"
pure "ahoj"
:t return "ahoj"
:t pure "ahoj"

"ahoj"

"ahoj"

return "ahoj" :: forall {m :: * -> *}. Monad m => m String

pure "ahoj" :: forall {f :: * -> *}. Applicative f => f String

In [55]:
putStrLn "Vengo a empeñar a mi hijo."
print (2 ^ 10)

Vengo a empeñar a mi hijo.

1024

Pokud chceme akce řetězit, používáme nejčastěji `do` notaci (což je jen pohodlnější zápis pro bind `(>>=)`).

In [56]:
import Data.Char (toUpper)
import Text.Read (readMaybe)

In [57]:
greet :: IO ()
greet = do
    putStrLn "¿Cómo te llamas?"
    name <- getLine
    putStrLn ("Se llama " ++ name ++ "!")

In [58]:
withStdin "Patricio\n" greet

¿Cómo te llamas?
Se llama Patricio!

Totéž bychom mohli napsat i „monadičtěji“ bez `do`:

In [59]:
greet :: IO ()
greet =
    putStrLn "¿Cómo te llamas?" >>
    getLine >>= \name ->
    putStrLn ("Se llama " ++ name ++ "!")

In [60]:
withStdin "Leo\n" greet

¿Cómo te llamas?
Se llama Leo!

Tenhle zápis je méně pohodlný, ale připomíná, že `IO` je opravdu další instance téhož vzoru, který už jste viděli u `Maybe` a `Gen`.

**Otázka:** Napište akci `echoUpper`, která:

1. vyzve uživatele k zadání řádku,
2. načte jej,
3. vypíše jej velkými písmeny,
4. a nakonec vypíše délku zadaného řádku.

In [61]:
echoUpper :: IO ()

echoUpper = do
    -- ?

: 

In [62]:
withStdin "ayuda me van a empeñar\n" echoUpper

: 

Všimněte si rozdílu mezi těmito řádky:
```haskell
line <- getLine
let upperLine = map toUpper line
```

V prvním případě „spouštíme“ akci (`getLine`), ve druhém jen čistě transformujeme už získanou hodnotu.

Někdy nechceme výsledek hned vytisknout, ale jen ho vrátit jako hodnotu uvnitř `IO`.

**Otázka:** Napište akci `askName`, která se zeptá na jméno a vrátí připravený pozdrav jako `IO String`. Sama jej tedy ještě nevypisuje.

In [63]:
askName :: IO String
askName = do
    -- ?

: 

In [64]:
withStdin "Patricio" $ askName >>= putStrLn

: 

Klasická začátečnická chyba při práci s `IO` vypadá nějak takto:

```haskell
do
    x = getLine 
    y = read x + 1 --
```

Jenže typem `getLine` není `String`, nýbrž **`IO String`**. Dokud tu akci „nespustíte“ pomocí `<-` uvnitř `do`, nemáte k dispozici ten skutečný řetězec (je to jen reprezentace potenciálního budoucího řetězce!).

Další praktický problém: funkce `read` je *partial* – když dostane špatný vstup, program spadne. Pro interaktivní programy je proto lepší používat bezpečnější `readMaybe`.

In [65]:
import Text.Read (readMaybe)

readIntMay :: IO (Maybe Int)
readIntMay = do
    line <- getLine
    return (readMaybe line)

In [66]:
readIntMay :: IO (Maybe Int)
readIntMay = readMaybe <$> getLine

In [67]:
withStdin "asdsad" readIntMay
withStdin "11685" readIntMay

Nothing

Just 11685

Povšimněte si typu: `IO (Maybe Int)`

Vyjadřuje kombinaci dvou „efektů“ s nějakým kontextem:
- `IO`: musím sáhnout do světa a něco načíst,
- `Maybe`: převod na `Int` může selhat.

In [68]:
sumTwoInputs :: IO ()
sumTwoInputs = do
    putStrLn "Zadejte první celé číslo:"
    mx <- readIntMay
    putStrLn "Zadejte druhé celé číslo:"
    my <- readIntMay
    case liftA2 (+) mx my of
        Just s  -> putStrLn ("Součet je " ++ show s)
        Nothing -> putStrLn "Aspoň jeden vstup nebyl platné celé číslo."

In [69]:
sumTwoInputs :: IO ()
sumTwoInputs = do
    putStrLn "Zadejte první celé číslo:"
    mx <- readIntMay
    putStrLn "Zadejte druhé celé číslo:"
    my <- readIntMay
    case (+) <$> mx <*> my of
        Just s  -> putStrLn ("Součet je " ++ show s)
        Nothing -> putStrLn "Aspoň jeden vstup nebyl platné celé číslo."

In [70]:
sumTwoInputs :: IO ()
sumTwoInputs = do
    putStrLn "Zadejte první celé číslo:"
    s1 <- getLine
    putStrLn "Zadejte druhé celé číslo:"
    s2 <- getLine
    case (readMaybe s1 :: Maybe Int, readMaybe s2 :: Maybe Int) of
        (Just x, Just y) -> putStrLn ("Součet je " ++ show (x + y))
        _                -> pure ()

In [71]:
readInt :: String -> Either String Int
readInt s =
    maybe (Left ("Neplatné číslo: " ++ show s)) Right (readMaybe s)

sumTwoInputs :: IO ()
sumTwoInputs = do
    s1 <- getLine
    s2 <- getLine
    case (readInt s1, readInt s2) of
        (Right x, Right y) -> putStrLn ("Součet je " ++ show (x + y))
        (Left e, _)        -> putStrLn e
        (_, Left e)        -> putStrLn e

In [72]:
withStdin "1564\n765\n" sumTwoInputs

Součet je 2329

Když máme **seznam akcí**, často je chceme provést jednu po druhé. K tomu slouží funkce jako:

- `sequence  :: [IO a] -> IO [a]`
- `sequence_ :: [IO a] -> IO ()`
- `mapM      :: (a -> IO b) -> [a] -> IO [b]`
- `mapM_     :: (a -> IO b) -> [a] -> IO ()`

Varianta s podtržítkem (`_`) zahazuje výsledky, což je velmi časté třeba u výpisů.

In [73]:
sequence_ [putStrLn "první řádek", putStrLn "druhý řádek", putStrLn "třetí řádek"]

první řádek
druhý řádek
třetí řádek

In [74]:
mapM_ print [x * x | x <- [1..5]]

1
4
9
16
25

In [75]:
printSquares :: Int -> IO ()
printSquares n = mapM_ (\x -> putStrLn (show x ++ " -> " ++ show (x * x))) [1..n]

In [76]:
-- Kontrola
printSquares 6

1 -> 1
2 -> 4
3 -> 9
4 -> 16
5 -> 25
6 -> 36